# 00 — Verificação do ambiente

Rode todas as células (**Run All**). Se tudo terminar sem erro e a figura aparecer, o ambiente está pronto para a Fase 1.

Pré-requisitos: `.venv` ativo como kernel, `pip install -e .` feito e `python scripts/download_data.py` executado.

In [ ]:
import sys
import cv2, numpy as np, pandas as pd, matplotlib
print('Python  ', sys.version.split()[0])
for m in (np, cv2, pd, matplotlib):
    print(f'{m.__name__:<12}', m.__version__)

In [ ]:
from pdi_defeitos.config import load_config
from pdi_defeitos.io import list_samples, load_sample

cfg = load_config()
print('Raiz do repositório:', cfg['root'])
print('Pasta da categoria: ', cfg['category_dir'])

## Contagem de imagens por conjunto e tipo

In [ ]:
df = list_samples(cfg['category_dir'])
df.groupby(['split', 'defect_type']).size().rename('n_imagens').to_frame()

## Uma imagem de cada tipo, com a máscara GT em vermelho

As máscaras são redimensionadas com `INTER_NEAREST`; a última linha confere que continuam binárias.

In [ ]:
import matplotlib.pyplot as plt

size = cfg['image']['size']
exemplos = df.groupby('defect_type').head(1).reset_index(drop=True)
fig, axes = plt.subplots(1, len(exemplos), figsize=(4 * len(exemplos), 4))
for ax, (_, row) in zip(np.atleast_1d(axes), exemplos.iterrows()):
    img, mask = load_sample(row, size=size)
    overlay = img.copy()
    overlay[mask == 1] = (0.5 * overlay[mask == 1] + [127, 0, 0]).astype(np.uint8)
    ax.imshow(overlay)
    ax.set_title(f"{row.defect_type}\n{int(mask.sum())} px de defeito")
    ax.axis('off')
plt.tight_layout()
plt.show()
print('Valores na máscara:', np.unique(mask))